# cluefin-dart-notes — 공시 원문에서 사업의 내용과 주석 노트 꺼내기

DART 공시 원문(dart4 XML)을 내려받아 `cluefin-dart-notes`로 읽는 예제입니다.
XBRL에 없는 서술형 내용(사업의 내용, 주석 설명)과 주석 노트의 표를 다룹니다.

다루는 내용: 최근 정기보고서(없으면 감사보고서) 찾기 → 원문 ZIP의 XML 전부 받기 → 원문 정리 기록 → 섹션 트리 →
사업의 개요 → 주석 노트 목차 → 노트 표를 DataFrame으로 → 첨부 감사보고서 요약값.

## 사전 준비

- 저장소 루트의 `.env.test`에 `DART_AUTH_KEY`가 있어야 합니다.
- 분석할 회사의 DART 고유번호(8자리)를 `DART_CORP_CODE` 환경 변수로 주거나 아래 셀에서 직접 넣으세요.
  고유번호는 `PublicDisclosure.corp_code()`로 찾을 수 있습니다(비상장회사는 종목코드가 비어 있습니다).
- 비상장회사도 됩니다. 사업·반기보고서를 내는 회사는 정기보고서로, 감사보고서만 내는 외감법인은 감사보고서로 진행합니다.
- 실행: `uv run --with jupyter --with pandas jupyter lab packages/cluefin-dart-notes/examples/dart_notes_analysis.ipynb`

## 1. DART 클라이언트

In [ ]:
import os
import tempfile
from pathlib import Path

import dotenv
import pandas as pd

from cluefin_openapi.dart._client import Client as DartClient
from cluefin_openapi.dart._public_disclosure import PublicDisclosure

dotenv.load_dotenv(dotenv.find_dotenv(".env.test"))
public_disclosure = PublicDisclosure(DartClient(auth_key=os.environ["DART_AUTH_KEY"]))

CORP_CODE = os.environ.get("DART_CORP_CODE", "")  # 예: "00000000"
assert len(CORP_CODE) == 8, "DART_CORP_CODE 에 고유번호 8자리를 넣으세요"

## 2. 최근 공시 원문 받기

정기공시(`pblntf_ty="A"`) 중 가장 최근 것을 고르고, 정기공시가 없으면 외부감사 공시(`pblntf_ty="F"`, 감사보고서·연결감사보고서)로
다시 찾습니다. 감사보고서만 내는 비상장 외감법인이 여기에 해당합니다. 감사보고서와 연결감사보고서는 접수번호가 다른 별개 공시입니다.

`disclosure_document_files`는 ZIP에 든 XML을 모두 저장하고 본문을 첫 번째로 돌려줍니다. 사업·반기보고서에는 감사(검토)보고서가
첨부로 함께 들어 있기도 하고, 감사보고서 공시의 ZIP에는 감사보고서 파일 하나만 있습니다.

In [ ]:
def latest_filing(pblntf_ty: str):
    found = public_disclosure.public_disclosure_search(
        corp_code=CORP_CODE, pblntf_ty=pblntf_ty, bgn_de="20240101", end_de="20261231", page_count=20
    ).result.list or []
    return max(found, key=lambda item: item.rcept_dt) if found else None


latest = latest_filing("A") or latest_filing("F")  # 정기공시가 없으면 감사보고서
assert latest is not None, "최근 정기공시·감사보고서가 없습니다. 검색 기간을 넓혀 보세요"
print(latest.report_nm, latest.rcept_dt)

work_dir = Path(tempfile.mkdtemp(prefix="dart-notes-"))
paths = public_disclosure.disclosure_document_files(latest.rcept_no, destination=work_dir)
[path.name for path in paths]

## 3. 문서 읽기와 원문 정리 기록

원문은 그대로 XML 파서에 넣으면 실패합니다(틀린 인코딩 선언, `R&D`의 `&`, 본문의 `<꺾쇠>` 등).
`load_document`가 정리 후 파싱하고, 무엇을 고쳤는지 `repair.reports`에 남깁니다.
`used_fallback`이 참이면 규칙이 모르는 새 결함이 나온 것입니다.

In [ ]:
from cluefin_dart_notes import load_document

doc = load_document(paths[0])
print(doc.document_name, doc.document_code, doc.company_name, doc.formula_version, doc.encoding)
print("위치 기반 수리 사용:", doc.repair.used_fallback, "| 경고:", doc.repair.warnings)
pd.DataFrame([(r.rule, r.count, r.samples[0].context if r.samples else "") for r in doc.repair.reports],
             columns=["규칙", "고친 수", "견본"])

## 4. 섹션 트리

In [ ]:
for section in doc.iter_sections():
    if section.level <= 2:
        print("    " * (section.level - 1) + section.title, f"[{section.assoc_code}]" if section.assoc_code else "")

## 5. 사업의 개요

`business_overviews`는 "II. 사업의 내용"의 "사업의 개요"를 업종 템플릿마다 모두 돌려줍니다(예: "(제조서비스업)사업의 개요"와
"(금융업)사업의 개요"). 오래된 공시는 하위 섹션이 없어 장 하나를 돌려줍니다. 감사보고서에는 사업의 내용이 없어 빈 목록입니다.

In [ ]:
from cluefin_dart_notes import Paragraph, business_overviews

overviews = business_overviews(doc)
if not overviews:
    print(f"{doc.document_name}에는 사업의 내용이 없습니다(감사보고서). 아래 주석과 요약값으로 진행합니다.")
for overview in overviews:
    paragraphs = [block.text for block in overview.blocks if isinstance(block, Paragraph)]
    print(f"## {overview.title} (문단 {len(paragraphs)}개)")
    print("\n\n".join(paragraphs[:3]), end="\n\n")

## 6. 주석 노트 목차

`split_notes`는 분할 방식과 경고를 함께 돌려줍니다. 최근 상장사 본문은 노트별 소제목으로(`heading`),
소제목이 없는 문서는 문단 번호를 따라가며(`sequence`) 나눕니다.

In [ ]:
from cluefin_dart_notes import Table, notes_sections, split_notes

for section, basis in notes_sections(doc):
    result = split_notes(section, basis)
    print(section.title, basis, result.method, f"노트 {len(result.notes)}개", result.warnings)

notes = [note for section, basis in notes_sections(doc) for note in split_notes(section, basis).notes]
pd.DataFrame(
    [(n.basis, n.number, n.title, sum(isinstance(b, Table) and b.bordered for b in n.blocks)) for n in notes],
    columns=["기준", "번호", "제목", "표 수"],
).head(40)

## 7. 노트의 표를 DataFrame으로

병합 셀은 `to_rows()`에서 펼쳐져 반복됩니다. 단위(`unit`)는 앞쪽 캡션에서 찾은 문자열이고 값을 환산하지는 않습니다.
`parse_amount`는 `(1,234)`·`△1,234`를 음수로 읽습니다.

In [ ]:
from cluefin_dart_notes import parse_amount

# 연결 주석을 먼저 찾고, 없으면(별도 감사보고서 등) 별도 주석에서 찾는다
note = next(n for n in sorted(notes, key=lambda n: n.basis != "consolidated") if "유형자산" in n.title)
table = next(b for b in note.blocks if isinstance(b, Table) and b.bordered)
print(note.number, note.title, "| 캡션:", table.caption, "| 단위:", table.unit)

rows = table.to_rows()
frame = pd.DataFrame(rows[1:], columns=[f"{i}:{name}" for i, name in enumerate(rows[0])])
frame

In [ ]:
numeric = frame.set_index(frame.columns[0]).apply(lambda column: column.map(parse_amount))
numeric

## 8. 감사보고서 요약값

감사보고서(`00760` 감사, `00761` 연결감사)의 `summary`에는 `SUMMARY/EXTRACTION` 값(감사인 고유번호, 감사의견 코드,
자산·부채·매출 총액 등)이 들어 있습니다. 정기보고서는 첨부 파일에서, 감사보고서 공시는 문서 자체에서 읽습니다.
감사보고서에는 재무제표 본표가 "(첨부)재무제표" 섹션(`D-0-0-0-0`)에 있습니다.

In [ ]:
from cluefin_dart_notes import find_section

audits = [doc] if doc.is_audit_report else [load_document(path) for path in paths[1:]]
for audit in audits:
    print(audit.document_name, dict(audit.summary))
    statements = find_section(audit, assoc_code="D-0-0-0-0")
    if statements is not None:
        print("  재무제표 본표:", [t.caption[:20] for t in statements.blocks if isinstance(t, Table) and t.bordered and t.caption])
if not audits:
    print("첨부된 감사보고서가 없습니다. 반기·분기보고서에는 보통 없고, 사업보고서에 첨부됩니다.")